# The MS COCO classification challenge

Razmig Kéchichian

This notebook defines the multi-label classification challenge on the [MS COCO dataset](https://cocodataset.org/). It defines the problem, sets the rules of organization and presents tools you are provided with to accomplish the challenge.


## 1. Problem statement

Each image has **several** categories of objects to predict, hence the difference compared to the classification problem we have seen on the CIFAR10 dataset where each image belonged to a **single** category, therefore the network loss function and prediction mechanism (only highest output probability) were defined taking this constraint into account.

We adapted the MS COCO dataset for the requirements of this challenge by, among other things, reducing the number of images and their dimensions to facilitate processing.

In the `ms-coco` dataset directory on the USB key you will find two sub-directories:
- `images`: which contains the images in train (65k) and test (~5k) subsets,
- `labels`: which lists labels for each of the images in the train subset only.

Each label file gives a list of class IDs that correspond to the class index in the following tuple:

## 0. Colab Environment Setup

1. Select **Runtime → Change runtime type → GPU** (T4 or better)
2. Upload the data to Google Drive. (**Uploading it as a single zip is recommended** — reading tens of thousands of images directly from Drive is very slow)
   - Option A (recommended): `MyDrive/ms-coco.zip` (the archive should contain `images/`, `labels/` or `ms-coco/images/`, `ms-coco/labels/`)
   - Option B: the `MyDrive/ms-coco/` folder as-is (containing `images/` and `labels/`)
3. The cell below mounts Drive and copies the data to the Colab local disk (`/content/data`).
4. Checkpoints and the submission JSON are **saved to Drive** so they survive session disconnects.

In [ ]:
# ===== Colab environment setup =====
!pip install -q timm

import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import torch
from google.colab import drive

In [ ]:
# Check GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: This is not a GPU runtime. Please change the runtime type to GPU.")

drive.mount('/content/drive')

CUDA available: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---- Path settings (edit to match your own Drive layout) ----
DRIVE_ZIP  = Path('/content/drive/MyDrive/ms-coco.zip')       # Option A: zip file
DRIVE_DATA = Path('/content/drive/MyDrive/ms-coco')           # Option B: folder
OUTPUT_DIR = Path('/content/drive/MyDrive/vit_coco_output_v2')   # Where v2 results are saved (v1 checkpoints stay untouched in vit_coco_output)
LOCAL_ROOT = Path('/content/data')                            # Local disk (fast, deleted when the session ends)

In [ ]:
!ls -l /content/drive/MyDrive/ms-coco.zip /content/ms-coco.zip

-rw------- 1 root root 1836661512 Sep 28 20:19 /content/drive/MyDrive/ms-coco.zip
-rw-r--r-- 1 root root  664666112 Sep 29 17:45 /content/ms-coco.zip


In [ ]:
import time

def robust_copy(src, dst, chunk=64 * 1024 * 1024, retries=20):
    """Resume copying from where it left off even if the Drive mount drops, and verify the size at the end."""
    src, dst = str(src), str(dst)
    total = os.path.getsize(src)
    for attempt in range(retries + 1):
        done = os.path.getsize(dst) if os.path.exists(dst) else 0
        if done >= total:
            break
        try:
            with open(src, 'rb') as fi, open(dst, 'ab' if done else 'wb') as fo:
                fi.seek(done)
                while True:
                    buf = fi.read(chunk)
                    if not buf:
                        break
                    fo.write(buf)
        except OSError as e:
            done = os.path.getsize(dst) if os.path.exists(dst) else 0
            print(f"Disconnected ({done/1e9:.2f}/{total/1e9:.2f} GB), retry {attempt + 1}: {e}")
            time.sleep(5)
    assert os.path.getsize(dst) == total, "The copy is incomplete"


# ---- Copy Drive -> local disk (skipped if already copied) ----
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
done_marker = LOCAL_ROOT / '.copy_done'

if done_marker.exists():
    print("Data already exists locally. Skipping the copy.")
else:
    if DRIVE_ZIP.exists():
        local_zip = Path('/content/ms-coco.zip')
        print("Copying zip to local disk...")
        robust_copy(DRIVE_ZIP, local_zip)          # Always verify size; resume if disconnected
        print("Extracting (local)...")
        subprocess.run(['unzip', '-q', '-o', str(local_zip), '-d', str(LOCAL_ROOT)], check=True)
        local_zip.unlink()                         # Delete only after successful extraction
    elif DRIVE_DATA.exists():
        print(f"Copying folder (this may take a while): {DRIVE_DATA}")
        shutil.copytree(DRIVE_DATA, LOCAL_ROOT / 'ms-coco', dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            f"Could not find the data on Drive.\n  zip: {DRIVE_ZIP}\n  folder: {DRIVE_DATA}\nPlease check the paths."
        )
    done_marker.touch()
    print("Copy complete")

# Check directory structure
for p in sorted(LOCAL_ROOT.rglob('*')):
    if p.is_dir() and len(p.relative_to(LOCAL_ROOT).parts) <= 3:
        print(p.relative_to(LOCAL_ROOT))

DATA_ROOT = LOCAL_ROOT                    # dataset_paths() automatically searches under root and root/ms-coco
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = min(8, os.cpu_count() or 2)  # Colab has few CPU cores (usually 2-8)
print("NUM_WORKERS =", NUM_WORKERS)

Copying zip to local disk...
Extracting (local)...
Copy complete
ms-coco
ms-coco/images
ms-coco/images/test
ms-coco/images/train
ms-coco/labels
ms-coco/labels/train
NUM_WORKERS = 2


In [ ]:
classes = ("person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
           "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
           "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
           "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
           "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
           "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
           "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
           "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
           "hair drier", "toothbrush")

Your goal is to follow a **transfer learning strategy** in training and validating a network on **your own distribution of training data into training and a validation subsets**, then to **test it on the test subset** by producing a [JSON file](https://en.wikipedia.org/wiki/JSON) with content of the following format:

```
{
    "000000000139": [
        56,
        60,
        62
    ],
    "000000000285": [
        21,
    ],
    "000000000632": [
        57,
        59,
    73
    ],
    # other test images
}
```

In this file, the name (without extension) of each test image is associated with a list of class indices predicted by your network. Make sure that the JSON file you produce **follows this format strictly**.

You will submit your JSON prediction file to the following [online evaluation server and leaderboard](https://www.creatis.insa-lyon.fr/kechichian/ms-coco-classif-leaderboard.html), which will evaluate your predictions on test set labels, unavailable to you.

<div class="alert alert-block alert-danger"> <b>WARNING:</b> Use this server with <b>the greatest care</b>. A new submission with identical Participant or group name will <b>overwrite</b> the identically named submission, if one already exists, therefore check the leaderboard first. <b>Do not make duplicate leaderboard entries for your group</b>, keep track of your test scores privately. Also pay attention to upload only JSON files of the required format.<br>
</div>

The evaluation server calculates and returns mean performances over all classes, and optionally per class performances. Entries in the leaderboard are sorted by the F1 metric.

You can request an evaluation as many times as you want. It is up to you to specify the final evaluation by updating the leaderboard entry corresponding to your Participant or group name. This entry will be taken into account for grading your work.

It goes without saying that it is **prohibited** to use another distribution of the MS COCO database for training, e.g. the Torchvision dataset.


## 2. Organization

- Given the scope of the project, you will work in groups of 2.
- Work on the challenge begins lab 3 session.
- Results are due exactly in **two weeks** after lab 3 session. They comrpise:
    - a submission to the leaderboard,
    - a commented Python script (with any necessary modules) or Jupyter Notebook, uploaded on Moodle in the challenge repository by one of the members of the group.
    
    
## 3. Tools and evaluation

In addition to the MS COCO annotated data and the evaluation server, we provide you with most code building blocks. Your task is to understand them and use them to create the glue logic, that is the main program, putting all these blocks together and completing them as necessary to implement a complete machine learning workflow to train and validate possibly several models, and produce the test JSON file to be submitted to the leaderboard. Comparative study of networks and analysis of results is of particular importance for grading. Please refer to course organization slides for the details of grading.

### 3.1 Custom `Dataset`s

We provide you with two custom `torch.utils.data.Dataset` sub-classes to use in training and testing.

In [ ]:
import os
from glob import glob
from pathlib import Path

from PIL import Image
import torch


class COCOTrainImageDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, annotations_dir, max_images=None, transform=None):
        self.img_labels = sorted(glob("*.cls", root_dir=annotations_dir))
        if max_images:
            self.img_labels = self.img_labels[:max_images]
        self.img_dir = img_dir
        self.annotations_dir = annotations_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, Path(self.img_labels[idx]).stem + ".jpg")
        labels_path = os.path.join(self.annotations_dir, self.img_labels[idx])
        image = Image.open(img_path).convert("RGB")
        with open(labels_path) as f:
            labels = [int(label) for label in f.readlines()]
        if self.transform:
            image = self.transform(image)
        labels = torch.zeros(80).scatter_(0, torch.tensor(labels), value=1)
        return image, labels


class COCOTestImageDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_list = sorted(glob("*.jpg", root_dir=img_dir))
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_list[idx])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, Path(img_path).stem # filename w/o extension

### 3.2 Training and validation loops

The following are two general-purpose classification train and validation loop functions to be called inside the epochs for-loop with appropriate argument settings.

Pay particular attention to the `validation_loop()` function's arguments `multi_label`, `th_multi_label` and `one_hot`. This function can be called on data loaders of both train and validation sets.

In [ ]:
import torch


def train_loop(train_loader, net, criterion, optimizer, device,
               mbatch_loss_group=-1):
    net.train()
    running_loss = 0.0
    mbatch_losses = []
    for i, data in enumerate(train_loader):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        # following condition False by default, unless mbatch_loss_group > 0
        if i % mbatch_loss_group == mbatch_loss_group - 1:
            mbatch_losses.append(running_loss / mbatch_loss_group)
            running_loss = 0.0
    if mbatch_loss_group > 0:
        return mbatch_losses


def validation_loop(val_loader, net, criterion, num_classes, device,
                    multi_label=False, th_multi_label=0.5, one_hot=False, class_metrics=False):
    net.eval()
    loss = 0
    correct = 0
    size = len(val_loader.dataset)
    class_total = {label:0 for label in range(num_classes)}
    class_tp = {label:0 for label in range(num_classes)}
    class_fp = {label:0 for label in range(num_classes)}
    with torch.no_grad():
        for data in val_loader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = net(images)
            loss += criterion(outputs, labels).item() * images.size(0)
            if not multi_label:
                predictions = torch.zeros_like(outputs)
                predictions[torch.arange(outputs.shape[0]), torch.argmax(outputs, dim=1)] = 1.0
            else:
                predictions = torch.where(outputs > th_multi_label, 1.0, 0.0)
            if not one_hot:
                labels_mat = torch.zeros_like(outputs)
                labels_mat[torch.arange(outputs.shape[0]), labels] = 1.0
                labels = labels_mat

            tps = predictions * labels
            fps = predictions - tps

            tps = tps.sum(dim=0)
            fps = fps.sum(dim=0)
            lbls = labels.sum(dim=0)

            for c in range(num_classes):
                class_tp[c] += tps[c]
                class_fp[c] += fps[c]
                class_total[c] += lbls[c]

            correct += tps.sum()

    class_prec = []
    class_recall = []
    freqs = []
    for c in range(num_classes):
        class_prec.append(0 if class_tp[c] == 0 else
                          class_tp[c] / (class_tp[c] + class_fp[c]))
        class_recall.append(0 if class_tp[c] == 0 else
                            class_tp[c] / class_total[c])
        freqs.append(class_total[c])

    freqs = torch.tensor(freqs)
    class_weights = 1. / freqs
    class_weights /= class_weights.sum()
    class_prec = torch.tensor(class_prec)
    class_recall = torch.tensor(class_recall)
    prec = (class_prec * class_weights).sum()
    recall = (class_recall * class_weights).sum()
    f1 = 2. / (1/prec + 1/recall)
    val_loss = loss / size
    accuracy = correct / freqs.sum()
    results = {"loss": val_loss, "accuracy": accuracy, "f1": f1,\
               "precision": prec, "recall": recall}

    if class_metrics:
        class_results = []
        for p, r in zip(class_prec, class_recall):
            f1 = (0 if p == r == 0 else 2. / (1/p + 1/r))
            class_results.append({"f1": f1, "precision": p, "recall": r})
        results = results, class_results

    return results

### 3.3 Tensorboard logging (optional)

Evaluation metrics and losses produced by the `validation_loop()` function on train and validation data can be logged to a [Tensorboard `SummaryWriter`](https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html) which allows you to observe training graphically via the following function:

In [ ]:
def update_graphs(summary_writer, epoch, train_results, test_results,
                  train_class_results=None, test_class_results=None,
                  class_names = None, mbatch_group=-1, mbatch_count=0, mbatch_losses=None):
    if mbatch_group > 0:
        for i in range(len(mbatch_losses)):
            summary_writer.add_scalar("Losses/Train mini-batches",
                                  mbatch_losses[i],
                                  epoch * mbatch_count + (i+1)*mbatch_group)

    summary_writer.add_scalars("Losses/Train Loss vs Test Loss",
                               {"Train Loss" : train_results["loss"],
                                "Test Loss" : test_results["loss"]},
                               (epoch + 1) if not mbatch_group > 0
                                     else (epoch + 1) * mbatch_count)

    summary_writer.add_scalars("Metrics/Train Accuracy vs Test Accuracy",
                               {"Train Accuracy" : train_results["accuracy"],
                                "Test Accuracy" : test_results["accuracy"]},
                               (epoch + 1) if not mbatch_group > 0
                                     else (epoch + 1) * mbatch_count)

    summary_writer.add_scalars("Metrics/Train F1 vs Test F1",
                               {"Train F1" : train_results["f1"],
                                "Test F1" : test_results["f1"]},
                               (epoch + 1) if not mbatch_group > 0
                                     else (epoch + 1) * mbatch_count)

    summary_writer.add_scalars("Metrics/Train Precision vs Test Precision",
                               {"Train Precision" : train_results["precision"],
                                "Test Precision" : test_results["precision"]},
                               (epoch + 1) if not mbatch_group > 0
                                     else (epoch + 1) * mbatch_count)

    summary_writer.add_scalars("Metrics/Train Recall vs Test Recall",
                               {"Train Recall" : train_results["recall"],
                                "Test Recall" : test_results["recall"]},
                               (epoch + 1) if not mbatch_group > 0
                                     else (epoch + 1) * mbatch_count)

    if train_class_results and test_class_results:
        for i in range(len(train_class_results)):
            summary_writer.add_scalars(f"Class Metrics/{class_names[i]}/Train F1 vs Test F1",
                                       {"Train F1" : train_class_results[i]["f1"],
                                        "Test F1" : test_class_results[i]["f1"]},
                                       (epoch + 1) if not mbatch_group > 0
                                             else (epoch + 1) * mbatch_count)

            summary_writer.add_scalars(f"Class Metrics/{class_names[i]}/Train Precision vs Test Precision",
                                       {"Train Precision" : train_class_results[i]["precision"],
                                        "Test Precision" : test_class_results[i]["precision"]},
                                       (epoch + 1) if not mbatch_group > 0
                                             else (epoch + 1) * mbatch_count)

            summary_writer.add_scalars(f"Class Metrics/{class_names[i]}/Train Recall vs Test Recall",
                                       {"Train Recall" : train_class_results[i]["recall"],
                                        "Test Recall" : test_class_results[i]["recall"]},
                                       (epoch + 1) if not mbatch_group > 0
                                             else (epoch + 1) * mbatch_count)
    summary_writer.flush()

## 4. The skeleton of the model training and validation program

Your main program should have more or less the following sections and control flow:

In [ ]:
from __future__ import annotations

import copy
import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
import timm
from timm.data import create_transform, resolve_data_config
from timm.optim import create_optimizer_v2

NUM_CLASSES = 80


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


class CocoTrainDataset(Dataset):
    def __init__(self, image_dir: Path, label_dir: Path, transform=None):
        self.image_dir, self.label_dir, self.transform = image_dir, label_dir, transform
        self.names = sorted(p.stem for p in label_dir.glob("*.cls"))
        if not self.names:
            raise FileNotFoundError(f"No .cls annotations found in {label_dir}")
        self.targets = torch.zeros((len(self.names), NUM_CLASSES), dtype=torch.float32)
        for row, name in enumerate(self.names):
            indices = [int(x) for x in (label_dir / f"{name}.cls").read_text().split()]
            if indices:
                self.targets[row, indices] = 1.0

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image = Image.open(self.image_dir / f"{name}.jpg").convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.targets[index]


class CocoTestDataset(Dataset):
    def __init__(self, image_dir: Path, transform):
        self.paths = sorted(image_dir.glob("*.jpg"))
        if not self.paths:
            raise FileNotFoundError(f"No JPG test images found in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        return self.transform(Image.open(path).convert("RGB")), path.stem


def dataset_paths(data_root: str | Path) -> tuple[Path, Path, Path]:
    root = Path(data_root)
    candidates = []
    for dataset_root in (root, root / "ms-coco", root / "ms_coco"):
        candidates.extend([
            # labels/train first (higher priority)
            (dataset_root / "images" / "train", dataset_root / "labels" / "train", dataset_root / "images" / "test"),
            (dataset_root / "images" / "train", dataset_root / "labels", dataset_root / "images" / "test"),
            (dataset_root / "images" / "train2017", dataset_root / "labels" / "train", dataset_root / "images" / "test"),
            (dataset_root / "train", dataset_root / "labels", dataset_root / "test"),
        ])
    for paths in candidates:
        if all(p.is_dir() for p in paths):
            return paths
    raise FileNotFoundError(
        f"Could not find images/train, labels/train and images/test below {root}."
    )


@dataclass(frozen=True)
class Trial:
    lr: float
    weight_decay: float
    pos_weight_strength: float
    drop_rate: float
    label_smoothing: float
    layer_decay: float = 0.75   # Layer-wise lr decay (LLRD): lower layers get smaller lr


# v1(0.6034): lr=2e-4, wd=0.05, pos_w=0.5, drop=0.1, ls=0.1, 8 epochs, same lr for all layers
# v2: start from the same values (keeping head/upper-layer lr) but lower only the lower-layer lr with LLRD, plus EMA and longer training
DEFAULT_TRIALS = (
    Trial(lr=2e-4, weight_decay=0.05, pos_weight_strength=0.5, drop_rate=0.1, label_smoothing=0.1, layer_decay=0.75),
    Trial(lr=1e-4, weight_decay=0.05, pos_weight_strength=0.5, drop_rate=0.1, label_smoothing=0.1, layer_decay=0.80),
)


def make_model(model_name: str, drop_rate: float, device: torch.device) -> nn.Module:
    return timm.create_model(
        model_name, pretrained=True, num_classes=NUM_CLASSES, drop_rate=drop_rate
    ).to(device)


def make_pos_weight(targets: torch.Tensor, strength: float) -> torch.Tensor | None:
    if strength == 0:
        return None
    positives = targets.sum(0).clamp_min(1)
    ratio = ((len(targets) - positives) / positives).clamp(max=20)
    return 1 + strength * (ratio - 1)


def macro_f1(probs: torch.Tensor, targets: torch.Tensor, thresholds: torch.Tensor) -> float:
    predictions = probs >= thresholds.unsqueeze(0)
    tp = (predictions & targets.bool()).sum(0).float()
    fp = (predictions & ~targets.bool()).sum(0).float()
    fn = (~predictions & targets.bool()).sum(0).float()
    valid = (tp + fp + fn) > 0
    f1 = torch.zeros(NUM_CLASSES)
    f1[valid] = 2 * tp[valid] / (2 * tp[valid] + fp[valid] + fn[valid])
    return f1.mean().item()


def calibrate_thresholds(probs: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, float]:
    # Wider and finer search: 0.05 ~ 0.85, step 0.01
    grid = torch.arange(0.05, 0.86, 0.01)
    thresholds = torch.full((NUM_CLASSES,), 0.35)
    for class_id in range(NUM_CLASSES):
        y = targets[:, class_id].bool()
        if y.sum() == 0:
            continue
        best_f1, best_t = -1.0, 0.35
        for threshold in grid:
            p = probs[:, class_id] >= threshold
            tp = (p & y).sum().float()
            score = (2 * tp / (p.sum() + y.sum()).clamp_min(1)).item()
            if score > best_f1:
                best_f1, best_t = score, float(threshold)
        thresholds[class_id] = best_t
    return thresholds, macro_f1(probs, targets, thresholds)


@torch.inference_mode()
def predict_validation(model, loader, device):
    model.eval()
    probs, targets = [], []
    for images, labels in loader:
        logits = model(images.to(device, non_blocking=True))
        probs.append(logits.sigmoid().cpu())
        targets.append(labels.cpu())
    return torch.cat(probs), torch.cat(targets)


def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, device):
    model.train()
    for images, labels in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            loss = criterion(model(images.to(device, non_blocking=True)), labels.to(device, non_blocking=True))
        scaler.scale(loss).backward()
        # gradient clipping: improves training stability
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()


class EMA:
    """Exponential moving average of weights. Using EMA weights for evaluation/inference usually raises the validation score more stably."""

    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.steps = decay, 0

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        self.steps += 1
        d = min(self.decay, (1 + self.steps) / (10 + self.steps))  # Use a lower decay early on so the EMA catches up quickly
        model_state = model.state_dict()
        for key, ema_value in self.module.state_dict().items():
            value = model_state[key].detach()
            if ema_value.dtype.is_floating_point:
                ema_value.mul_(d).add_(value, alpha=1 - d)
            else:
                ema_value.copy_(value)


def atomic_save(obj, path: Path) -> None:
    # Write to a temp file and then replace, so the existing file is not corrupted if the session drops mid-save
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def run_hyperparameter_search(
    data_root: str | Path,
    output_dir: str | Path,
    *,
    model_name: str = "vit_base_patch16_224.augreg_in21k_ft_in1k",
    tune_epochs: int = 15,
    batch_size: int = 16,
    num_workers: int = 4,
    validation_fraction: float = 0.12,
    trials: Iterable[Trial] = DEFAULT_TRIALS,
    seed: int = 42,
    ema_decay: float = 0.9998,
    save_every: int = 2,             # Save a resume checkpoint every N epochs
    best_name: str = "best_vit_v2.pt",
) -> Path:
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("WARNING: CUDA was not found; ViT tuning will be very slow on CPU.")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    image_dir, label_dir, _ = dataset_paths(data_root)

    probe = make_model(model_name, 0.0, device)
    data_config = resolve_data_config(probe.pretrained_cfg, model=probe)
    del probe
    if device.type == "cuda":
        torch.cuda.empty_cache()

    train_tf = create_transform(
        **data_config, is_training=True,
        auto_augment="rand-m9-mstd0.5-inc1",
        re_prob=0.25, re_mode="pixel",
    )
    eval_tf = create_transform(**data_config, is_training=False)

    base = CocoTrainDataset(image_dir, label_dir)
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(base), generator=generator).tolist()   # Same split as v1 (seed=42)
    split = int(len(base) * (1 - validation_fraction))
    train_indices, val_indices = indices[:split], indices[split:]
    train_set = Subset(CocoTrainDataset(image_dir, label_dir, train_tf), train_indices)
    val_set = Subset(CocoTrainDataset(image_dir, label_dir, eval_tf), val_indices)

    loader_args = dict(
        batch_size=batch_size, num_workers=num_workers,
        pin_memory=device.type == "cuda", persistent_workers=num_workers > 0
    )
    train_loader = DataLoader(train_set, shuffle=True, drop_last=True, **loader_args)
    val_loader = DataLoader(val_set, shuffle=False, **loader_args)
    pos_weight_base = make_pos_weight(base.targets[train_indices], 1.0)

    best_path = output_dir / best_name
    last_path = output_dir / "last_state.pt"
    best_score, summary, resume = -1.0, [], None

    # If the session was interrupted, resume training from the last saved point (delete last_state.pt to start fresh)
    if last_path.exists():
        resume = torch.load(last_path, map_location="cpu", weights_only=False)
        best_score, summary = resume["best_score"], resume["summary"]
        print(f"Resuming: trial {resume['trial_id']}, from after epoch {resume['epoch']} (best={best_score:.4f})")

    for trial_id, trial in enumerate(trials, start=1):
        if resume is not None and trial_id < resume["trial_id"]:
            continue
        print(f"\nTrial {trial_id}: {asdict(trial)}")
        model = make_model(model_name, trial.drop_rate, device)
        ema = EMA(model, decay=ema_decay)

        pos_weight = None if pos_weight_base is None else 1 + trial.pos_weight_strength * (pos_weight_base - 1)
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=None if pos_weight is None else pos_weight.to(device),
            reduction="mean",
        )
        label_smooth_eps = trial.label_smoothing

        # LLRD: create parameter groups with an lr_scale depending on layer depth (bias/norm/pos_embed are excluded from weight decay)
        optimizer = create_optimizer_v2(
            model, opt="adamw", lr=trial.lr, weight_decay=trial.weight_decay,
            layer_decay=trial.layer_decay, betas=(0.9, 0.999),
        )
        # torch's OneCycleLR does not know about lr_scale, so pass per-group max_lr explicitly
        max_lrs = [trial.lr * g.get("lr_scale", 1.0) for g in optimizer.param_groups]
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=max_lrs,
            epochs=tune_epochs, steps_per_epoch=len(train_loader),
            pct_start=0.15, div_factor=25, final_div_factor=1000
        )
        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

        start_epoch, trial_best = 1, -1.0
        if resume is not None and trial_id == resume["trial_id"]:
            model.load_state_dict(resume["model"])
            ema.module.load_state_dict(resume["ema"])
            ema.steps = resume["ema_steps"]
            optimizer.load_state_dict(resume["optimizer"])
            scheduler.load_state_dict(resume["scheduler"])
            scaler.load_state_dict(resume["scaler"])
            start_epoch, trial_best = resume["epoch"] + 1, resume["trial_best"]
            resume = None

        for epoch in range(start_epoch, tune_epochs + 1):
            model.train()
            for images, labels in train_loader:
                images = images.to(device, non_blocking=True)
                soft_labels = labels * (1 - label_smooth_eps) + label_smooth_eps / NUM_CLASSES
                soft_labels = soft_labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                    loss = criterion(model(images), soft_labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                ema.update(model)

            # Evaluate both the raw and EMA weights, and use the better one as that epoch's candidate
            results = {}
            for tag, m in (("raw", model), ("ema", ema.module)):
                probs, targets = predict_validation(m, val_loader, device)
                thresholds, score = calibrate_thresholds(probs, targets)
                results[tag] = (score, thresholds, m)
            tag = max(results, key=lambda k: results[k][0])
            score, thresholds, chosen = results[tag]
            print(f"  epoch {epoch}/{tune_epochs}  val macro-F1: raw={results['raw'][0]:.4f}  ema={results['ema'][0]:.4f}")

            trial_best = max(trial_best, score)
            if score > best_score:
                best_score = score
                atomic_save({
                    "model_name": model_name,
                    "drop_rate": trial.drop_rate,
                    "state_dict": chosen.state_dict(),
                    "thresholds": thresholds,
                    "score": score,
                    "weights": tag,
                    "trial": asdict(trial),
                    "data_config": data_config,
                }, best_path)
                print(f"  ★ New best saved ({tag}): macro-F1={best_score:.4f}")

            if epoch % save_every == 0 and epoch < tune_epochs:
                atomic_save({
                    "trial_id": trial_id, "epoch": epoch, "trial_best": trial_best,
                    "best_score": best_score, "summary": summary,
                    "model": model.state_dict(), "ema": ema.module.state_dict(), "ema_steps": ema.steps,
                    "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(),
                }, last_path)

        summary.append({"trial": asdict(trial), "best_val_macro_f1": trial_best})
        del model, ema, optimizer, scheduler
        if device.type == "cuda":
            torch.cuda.empty_cache()

    (output_dir / "search_results.json").write_text(json.dumps(summary, indent=2))
    print(f"\nBest single model: macro-F1={best_score:.4f}, checkpoint={best_path}")
    return best_path


@torch.inference_mode()
def create_submission(
    data_root: str | Path,
    checkpoint_path: str | Path,
    json_path: str | Path,
    *,
    batch_size=64,
    num_workers=4,
) -> Path:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = make_model(checkpoint["model_name"], checkpoint["drop_rate"], device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    _, _, test_dir = dataset_paths(data_root)
    eval_tf = create_transform(**checkpoint["data_config"], is_training=False)
    loader = DataLoader(
        CocoTestDataset(test_dir, eval_tf), batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=device.type == "cuda",
        persistent_workers=num_workers > 0,
    )
    thresholds = checkpoint["thresholds"].to(device)
    predictions = {}
    for images, names in loader:
        positive = model(images.to(device, non_blocking=True)).sigmoid() >= thresholds
        for name, labels in zip(names, positive.cpu()):
            predictions[name] = torch.where(labels)[0].tolist()
    json_path = Path(json_path)
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(predictions, indent=2))
    print(f"Wrote {len(predictions)} test predictions to {json_path}")
    return json_path


# ---- Run ----
# DATA_ROOT, OUTPUT_DIR, NUM_WORKERS are defined in the Colab setup cell above
BEST_CHECKPOINT = run_hyperparameter_search(
    data_root=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    model_name="vit_base_patch16_224.augreg_in21k_ft_in1k",
    tune_epochs=15,      # v1: 8 (the score was still rising at the end)
    batch_size=16,
    num_workers=NUM_WORKERS,
    validation_fraction=0.12,
    trials=DEFAULT_TRIALS[:1],   # use [:2] if time allows
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            


Trial 1: {'lr': 0.0002, 'weight_decay': 0.05, 'pos_weight_strength': 0.5, 'drop_rate': 0.1, 'label_smoothing': 0.1, 'layer_decay': 0.75}
  epoch 1/15  val macro-F1: raw=0.7238  ema=0.7415
  ★ New best saved (ema): macro-F1=0.7415
  epoch 2/15  val macro-F1: raw=0.7019  ema=0.7481
  ★ New best saved (ema): macro-F1=0.7481
  epoch 3/15  val macro-F1: raw=0.7120  ema=0.7449
  epoch 4/15  val macro-F1: raw=0.7113  ema=0.7479
  epoch 5/15  val macro-F1: raw=0.7143  ema=0.7498
  ★ New best saved (ema): macro-F1=0.7498
  epoch 6/15  val macro-F1: raw=0.7291  ema=0.7514
  ★ New best saved (ema): macro-F1=0.7514
  epoch 7/15  val macro-F1: raw=0.7298  ema=0.7555
  ★ New best saved (ema): macro-F1=0.7555
  epoch 8/15  val macro-F1: raw=0.7381  ema=0.7568
  ★ New best saved (ema): macro-F1=0.7568
  epoch 9/15  val macro-F1: raw=0.7433  ema=0.7577
  ★ New best saved (ema): macro-F1=0.7577
  epoch 10/15  val macro-F1: raw=0.7501  ema=0.7607
  ★ New best saved (ema): macro-F1=0.7607
  epoch 11/15  

focal loss

In [ ]:
# ============================================================
# Focal Loss experiment
# DATA_ROOT, OUTPUT_DIR, NUM_WORKERS,
# NUM_CLASSES, CocoTrainDataset, etc. must be defined in earlier cells
# ============================================================

import copy
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from torch.utils.data import DataLoader, Subset
from timm.data import create_transform, resolve_data_config
from timm.optim import create_optimizer_v2


# ------------------------------------------------------------
# 1. Settings
# ------------------------------------------------------------

MODEL_NAME = "vit_base_patch16_224.augreg_in21k_ft_in1k"

EPOCHS = 15
BATCH_SIZE = 16
VALIDATION_FRACTION = 0.12
SEED = 42

LR = 2e-4
WEIGHT_DECAY = 0.05
POS_WEIGHT_STRENGTH = 0.5
DROP_RATE = 0.1
LABEL_SMOOTHING = 0.1
LAYER_DECAY = 0.75

EMA_DECAY = 0.9998

# Key Focal Loss hyperparameter
GAMMA = 2.0

# Keep separate from the existing BCE checkpoint
BEST_CHECKPOINT = Path(OUTPUT_DIR) / "best_vit_focal.pt"


# ------------------------------------------------------------
# 2. Seed
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True


# ------------------------------------------------------------
# 3. Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("loss: Focal Loss")
print("gamma:", GAMMA)


# ------------------------------------------------------------
# 4. Focal Loss
# ------------------------------------------------------------

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer(
            "pos_weight",
            pos_weight if pos_weight is not None else torch.tensor(1.0)
        )

    def forward(self, logits, targets):

        # Compute BCE element-wise
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=self.pos_weight,
            reduction="none",
        )

        # sigmoid probability
        probs = torch.sigmoid(logits)

        # Probability assigned to the correct class
        p_t = (
            probs * targets
            + (1.0 - probs) * (1.0 - targets)
        )

        # Focal factor
        focal_factor = (1.0 - p_t).pow(self.gamma)

        loss = focal_factor * bce

        return loss.mean()


# ------------------------------------------------------------
# 5. Model
# ------------------------------------------------------------

def make_focal_model():
    return timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=NUM_CLASSES,
        drop_rate=DROP_RATE,
    ).to(device)


# ------------------------------------------------------------
# 6. Positive weight
# ------------------------------------------------------------

def make_pos_weight_focal(targets, strength):

    if strength == 0:
        return None

    positives = targets.sum(0).clamp_min(1)

    ratio = (
        (len(targets) - positives) / positives
    ).clamp(max=20)

    return 1 + strength * (ratio - 1)


# ------------------------------------------------------------
# 7. Validation metric
# ------------------------------------------------------------

def macro_f1_focal(probs, targets, thresholds):

    predictions = probs >= thresholds.unsqueeze(0)

    tp = (predictions & targets.bool()).sum(0).float()
    fp = (predictions & ~targets.bool()).sum(0).float()
    fn = (~predictions & targets.bool()).sum(0).float()

    valid = (tp + fp + fn) > 0

    f1 = torch.zeros(NUM_CLASSES)

    f1[valid] = (
        2 * tp[valid]
        / (2 * tp[valid] + fp[valid] + fn[valid])
    )

    return f1.mean().item()


def calibrate_thresholds_focal(probs, targets):

    grid = torch.arange(0.05, 0.86, 0.01)

    thresholds = torch.full(
        (NUM_CLASSES,),
        0.35
    )

    for class_id in range(NUM_CLASSES):

        y = targets[:, class_id].bool()

        if y.sum() == 0:
            continue

        best_f1 = -1.0
        best_t = 0.35

        for threshold in grid:

            p = probs[:, class_id] >= threshold

            tp = (p & y).sum().float()

            score = (
                2 * tp
                / (p.sum() + y.sum()).clamp_min(1)
            ).item()

            if score > best_f1:
                best_f1 = score
                best_t = float(threshold)

        thresholds[class_id] = best_t

    return (
        thresholds,
        macro_f1_focal(
            probs,
            targets,
            thresholds
        )
    )


@torch.inference_mode()
def predict_validation_focal(model, loader):

    model.eval()

    probs = []
    targets = []

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        logits = model(images)

        probs.append(
            logits.sigmoid().cpu()
        )

        targets.append(
            labels.cpu()
        )

    return (
        torch.cat(probs),
        torch.cat(targets)
    )


# ------------------------------------------------------------
# 8. EMA
# ------------------------------------------------------------

class EMA_Focal:

    def __init__(self, model, decay=0.9998):

        self.module = copy.deepcopy(model).eval()

        for p in self.module.parameters():
            p.requires_grad_(False)

        self.decay = decay
        self.steps = 0

    @torch.no_grad()
    def update(self, model):

        self.steps += 1

        d = min(
            self.decay,
            (1 + self.steps) / (10 + self.steps)
        )

        model_state = model.state_dict()

        for key, ema_value in self.module.state_dict().items():

            value = model_state[key].detach()

            if ema_value.dtype.is_floating_point:

                ema_value.mul_(d).add_(
                    value,
                    alpha=1 - d
                )

            else:
                ema_value.copy_(value)


# ------------------------------------------------------------
# 9. Dataset / split
# ------------------------------------------------------------

image_dir, label_dir, _ = dataset_paths(DATA_ROOT)

# Input config of the pretrained model
probe = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES,
).to(device)

data_config = resolve_data_config(
    probe.pretrained_cfg,
    model=probe
)

del probe

if device.type == "cuda":
    torch.cuda.empty_cache()


train_tf = create_transform(
    **data_config,
    is_training=True,
    auto_augment="rand-m9-mstd0.5-inc1",
    re_prob=0.25,
    re_mode="pixel",
)

eval_tf = create_transform(
    **data_config,
    is_training=False,
)


# Same dataset as the original
base = CocoTrainDataset(
    image_dir,
    label_dir
)


# Same seed=42 split as the original
generator = torch.Generator().manual_seed(SEED)

indices = torch.randperm(
    len(base),
    generator=generator
).tolist()

split = int(
    len(base) * (1 - VALIDATION_FRACTION)
)

train_indices = indices[:split]
val_indices = indices[split:]


train_set = Subset(
    CocoTrainDataset(
        image_dir,
        label_dir,
        train_tf
    ),
    train_indices
)

val_set = Subset(
    CocoTrainDataset(
        image_dir,
        label_dir,
        eval_tf
    ),
    val_indices
)


loader_args = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)

train_loader = DataLoader(
    train_set,
    shuffle=True,
    drop_last=True,
    **loader_args
)

val_loader = DataLoader(
    val_set,
    shuffle=False,
    **loader_args
)


print()
print("Train:", len(train_set))
print("Validation:", len(val_set))


# ------------------------------------------------------------
# 10. Positive weight
# ------------------------------------------------------------

pos_weight_base = make_pos_weight_focal(
    base.targets[train_indices],
    POS_WEIGHT_STRENGTH
)

if pos_weight_base is not None:
    pos_weight = pos_weight_base.to(device)
else:
    pos_weight = None


# ------------------------------------------------------------
# 11. Model / EMA / Loss
# ------------------------------------------------------------

model = make_focal_model()

ema = EMA_Focal(
    model,
    decay=EMA_DECAY
)


criterion = FocalLoss(
    gamma=GAMMA,
    pos_weight=pos_weight
).to(device)


# ------------------------------------------------------------
# 12. Optimizer
# ------------------------------------------------------------

optimizer = create_optimizer_v2(
    model,
    opt="adamw",
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    layer_decay=LAYER_DECAY,
    betas=(0.9, 0.999),
)


max_lrs = [
    LR * g.get("lr_scale", 1.0)
    for g in optimizer.param_groups
]


scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lrs,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    div_factor=25,
    final_div_factor=1000,
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=device.type == "cuda"
)


# ------------------------------------------------------------
# 13. Training
# ------------------------------------------------------------

best_score = -1.0

print()
print("=" * 60)
print("Start Focal Loss training")
print("=" * 60)


for epoch in range(1, EPOCHS + 1):

    model.train()

    running_loss = 0.0
    num_batches = 0

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        # Same label smoothing as the original BCE training
        soft_labels = (
            labels * (1 - LABEL_SMOOTHING)
            + LABEL_SMOOTHING / NUM_CLASSES
        )

        soft_labels = soft_labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            enabled=device.type == "cuda"
        ):

            logits = model(images)

            loss = criterion(
                logits,
                soft_labels
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        ema.update(model)

        running_loss += loss.item()
        num_batches += 1


    avg_loss = running_loss / num_batches


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    results = {}

    for tag, eval_model in (
        ("raw", model),
        ("ema", ema.module),
    ):

        probs, targets = predict_validation_focal(
            eval_model,
            val_loader
        )

        thresholds, score = calibrate_thresholds_focal(
            probs,
            targets
        )

        results[tag] = (
            score,
            thresholds,
            eval_model
        )


    raw_score = results["raw"][0]
    ema_score = results["ema"][0]

    tag = max(
        results,
        key=lambda x: results[x][0]
    )

    score, thresholds, chosen = results[tag]


    current_lr = optimizer.param_groups[0]["lr"]


    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"loss={avg_loss:.4f} | "
        f"lr={current_lr:.2e} | "
        f"raw F1={raw_score:.4f} | "
        f"ema F1={ema_score:.4f}"
    )


    # --------------------------------------------------------
    # Best checkpoint
    # --------------------------------------------------------

    if score > best_score:

        best_score = score

        torch.save(
            {
                "model_name": MODEL_NAME,
                "drop_rate": DROP_RATE,
                "state_dict": chosen.state_dict(),
                "thresholds": thresholds,
                "score": score,
                "weights": tag,

                "loss": "focal",
                "gamma": GAMMA,

                "trial": {
                    "lr": LR,
                    "weight_decay": WEIGHT_DECAY,
                    "pos_weight_strength": POS_WEIGHT_STRENGTH,
                    "drop_rate": DROP_RATE,
                    "label_smoothing": LABEL_SMOOTHING,
                    "layer_decay": LAYER_DECAY,
                },

                "data_config": data_config,
            },
            BEST_CHECKPOINT
        )

        print(
            f"  ★ Best saved: "
            f"macro-F1={best_score:.4f}"
        )


# ------------------------------------------------------------
# 14. Finish
# ------------------------------------------------------------

print()
print("=" * 60)
print("Focal Loss training finished")
print("=" * 60)
print(f"Best validation Macro-F1: {best_score:.4f}")
print(f"Checkpoint: {BEST_CHECKPOINT}")

## 5. The skeleton of the test submission program

This, much simpler, program should have the following sections and control flow:

In [ ]:
 # ---- Test inference and strict JSON submission ----
SUBMISSION_JSON = create_submission(
    data_root=DATA_ROOT, checkpoint_path=BEST_CHECKPOINT,
    json_path=OUTPUT_DIR / 'submission_vit_v2.json',
    batch_size=128, num_workers=NUM_WORKERS,
)
SUBMISSION_JSON

Wrote 4952 test predictions to /content/drive/MyDrive/vit_coco_output_v2/submission_vit_v2.json


PosixPath('/content/drive/MyDrive/vit_coco_output_v2/submission_vit_v2.json')